In [ ]:
#| default_exp core

# Core

> Shared notebook primitives for cell parsing, directives, chapters, and source ownership.

This module builds a small vocabulary on top of `fastcore.nbio` so readers, checkers, and editors share one representation.

In [ ]:
#| export
import ast, hashlib, os, re, tomllib
from dataclasses import dataclass, field
from pathlib import Path

from fastcore.nbio import mk_cell, read_nb

In [ ]:
from tempfile import TemporaryDirectory
from fastcore.nbio import new_nb, write_nb
from fastcore.test import test_eq

## Cell source and directives

Cells store source as either a string or a list of lines. `cell_source` normalizes both into one string, and `source_hash` creates a stable SHA-256 fingerprint for optimistic locking and execution tracking.

In [ ]:
#| export
_DIRECTIVE_RE = re.compile(r"^#\|\s*([A-Za-z0-9_-]+)(?:(?:\s*:\s*|\s+)(.*))?$")
_EXPORT_KEYS = frozenset({"export", "exports", "exporti"})

def cell_source(cell) -> str:
    "Return a cell's source as a single string."
    src = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    return "".join(src) if isinstance(src, list) else str(src or "")

def source_hash(text: str, length: int | None = 12) -> str:
    "Return a SHA-256 hex fingerprint of `text`, truncated to `length` chars unless `None`."
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return digest if length is None else digest[:length]

nbdev directives can live in leading `#|` comments or inside `cell.metadata.nbdev`. `cell_directives` merges both with source lines taking precedence, and `strip_directives` removes the directive header.

In [ ]:
#| export
def cell_directives(cell) -> dict[str, str]:
    "Merge nbdev directives from cell metadata and leading `#|` lines (source wins)."
    meta = (cell.get("metadata", {}) if isinstance(cell, dict) else getattr(cell, "metadata", {})) or {}
    dirs = {str(k): str(v) for k, v in (meta.get("nbdev") or {}).items()} if isinstance(meta, dict) else {}
    for line in cell_source(cell).splitlines():
        s = line.strip()
        if not s: continue
        m = _DIRECTIVE_RE.match(s)
        if not m: break
        dirs[m.group(1)] = (m.group(2) or "true").strip()
    return dirs

def strip_directives(src: str) -> str:
    "Remove leading `#|` directive lines while keeping the rest of `src` untouched."
    lines = src.splitlines()
    idx = 0
    while idx < len(lines) and (not lines[idx].strip() or _DIRECTIVE_RE.match(lines[idx].strip())): idx += 1
    return "\n".join(lines[idx:])

Export helpers distinguish `#| export`, `#| exports`, `#| exporti`, and explicit `#| export <module>` overrides, alongside predicates for visibility and execution.

In [ ]:
#| export
def cell_export_kind(cell) -> str:
    "Return `'export'`, `'exports'`, `'exporti'`, or `''` for a cell."
    if cell.get("cell_type") != "code": return ""
    dirs = cell_directives(cell)
    for key in ("export", "exports", "exporti"):
        if key in dirs: return key
    return ""

def cell_export_module(cell, default_exp: str | None = None) -> str | None:
    "Return the target module name for an exported cell, respecting `#| export <mod>` overrides."
    kind = cell_export_kind(cell)
    if not kind: return None
    val = cell_directives(cell).get(kind, "").strip()
    if val and val.lower() not in ("true", "1"): return val.split()[0]
    return default_exp

In [ ]:
#| export
def is_exported(cell) -> bool:
    "True when `cell` exports library code (`export`, `exports`, or `exporti`)."
    return bool(cell_export_kind(cell))

def is_hidden(cell) -> bool:
    "True when `cell` carries `#| hide`."
    val = cell_directives(cell).get("hide", "").lower()
    return val not in ("", "false", "0")

def is_eval_false(cell) -> bool:
    "True when `cell` is skipped during execution via `#| eval: false`."
    return cell_directives(cell).get("eval", "").lower() == "false"

Both `#|` comments (space-separated or colon-separated) and metadata dictionaries resolve into the same directive map.

In [ ]:
c_src = mk_cell("#| default_exp calc\n#| export helpers\n#| eval: false\nx = 1")
c_meta = mk_cell("y = 2", metadata={"nbdev": {"export": "true", "hide": "true"}})
test_eq(cell_directives(c_src)["default_exp"], "calc")
test_eq((cell_export_kind(c_src), cell_export_module(c_src, "calc")), ("export", "helpers"))
test_eq(is_exported(c_src) and is_eval_false(c_src), True)
test_eq(is_exported(c_meta) and is_hidden(c_meta), True)
test_eq(strip_directives(cell_source(c_src)), "x = 1")
dict(src=cell_directives(c_src), meta=cell_directives(c_meta), hash=source_hash("x = 1"))

## Python structure in cells

Static checks report definitions and imports with their 1-based line numbers inside the cell.

In [ ]:
#| export
@dataclass(frozen=True)
class DefInfo:
    "One top-level Python definition inside a code cell."
    name: str
    kind: str  # 'function', 'async_function', 'class', or 'patch'
    lineno: int
    end_lineno: int
    receiver: str = ""  # Receiver class name for `@patch` methods
    is_public: bool = True
    bases: tuple[str, ...] = ()  # Base class names for `class` definitions
    delegates_to: str = ""  # Target callable from `@delegates(...)`
    export_kind: str = ""  # 'export', 'exports', 'exporti', or ''

@dataclass(frozen=True)
class ImportInfo:
    "One imported binding inside a code cell."
    module: str
    name: str
    alias: str
    lineno: int
    is_star: bool = False

Replacing leading `#|` directives and IPython line magics (`%...`, `!...`) with blank lines keeps AST line numbers aligned with the cell source. Cell magics (`%%bash`, `%%html`) return `None` instead of false syntax errors.

In [ ]:
#| export
def clean_python_source(src: str) -> str | None:
    "Blank out `#|` directives and line magics to preserve line numbers; return `None` for non-Python cell magics."
    lines = src.splitlines()
    for line in lines:
        s = line.strip()
        if not s or _DIRECTIVE_RE.match(s): continue
        if s.startswith("%%"): return None
        break
    out, in_header = [], True
    for line in lines:
        s = line.strip()
        if in_header and (not s or _DIRECTIVE_RE.match(s)):
            out.append("")
            continue
        in_header = False
        out.append("" if s.startswith(("%", "!")) else line)
    return "\n".join(out)

def parse_cell_ast(cell) -> tuple[ast.Module | None, SyntaxError | None]:
    "Parse a code cell into `(tree, syntax_error)`; non-code and magic cells return `(None, None)`."
    if cell.get("cell_type") != "code": return None, None
    cleaned = clean_python_source(cell_source(cell))
    if not cleaned or not cleaned.strip(): return None, None
    try: return ast.parse(cleaned), None
    except SyntaxError as exc: return None, exc

`cell_defs` extracts top-level functions, classes (including base classes), `@patch` methods (with their annotated receiver type), and `@delegates` targets. `cell_imports` lists every imported name and alias.

In [ ]:
#| export
def _is_patch_dec(d) -> bool:
    if isinstance(d, ast.Name): return d.id == "patch"
    if isinstance(d, ast.Call) and isinstance(d.func, ast.Name): return d.func.id == "patch"
    return isinstance(d, ast.Attribute) and d.attr == "patch"

def _patch_receiver(node: ast.FunctionDef | ast.AsyncFunctionDef) -> str:
    if not any(_is_patch_dec(d) for d in node.decorator_list) or not node.args.args: return ""
    ann = node.args.args[0].annotation
    return ast.unparse(ann) if ann is not None else "patched"

def _delegates_target(node: ast.FunctionDef | ast.AsyncFunctionDef | ast.ClassDef) -> str:
    for d in node.decorator_list:
        if not isinstance(d, ast.Call): continue
        fn = d.func
        is_del = (isinstance(fn, ast.Name) and fn.id == "delegates") or (isinstance(fn, ast.Attribute) and fn.attr == "delegates")
        if not is_del: continue
        if d.args: return ast.unparse(d.args[0])
        for kw in d.keywords:
            if kw.arg == "to": return ast.unparse(kw.value)
    return ""

In [ ]:
#| export
def cell_defs(cell) -> list[DefInfo]:
    "Return top-level function, class, and `@patch` definitions in `cell`."
    tree, _ = parse_cell_ast(cell)
    if tree is None: return []
    exp_kind = cell_export_kind(cell)
    defs = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            recv = _patch_receiver(node)
            kind = "patch" if recv else ("async_function" if isinstance(node, ast.AsyncFunctionDef) else "function")
            defs.append(DefInfo(node.name, kind, node.lineno, node.end_lineno or node.lineno,
                recv, not node.name.startswith("_"), (), _delegates_target(node), exp_kind))
        elif isinstance(node, ast.ClassDef):
            bases = tuple(ast.unparse(b) for b in node.bases)
            defs.append(DefInfo(node.name, "class", node.lineno, node.end_lineno or node.lineno,
                "", not node.name.startswith("_"), bases, _delegates_target(node), exp_kind))
    return defs

`cell_imports` extracts top-level `import` and `from ... import` bindings, preserving aliases and star imports.

In [ ]:
#| export
def cell_imports(cell) -> list[ImportInfo]:
    "Return top-level import bindings in `cell`."
    tree, _ = parse_cell_ast(cell)
    if tree is None: return []
    items = []
    for node in tree.body:
        if isinstance(node, ast.Import):
            for a in node.names: items.append(ImportInfo(a.name, a.name, a.asname or a.name.split(".", 1)[0], node.lineno))
        elif isinstance(node, ast.ImportFrom):
            mod = "." * node.level + (node.module or "")
            for a in node.names: items.append(ImportInfo(mod, a.name, a.asname or a.name, node.lineno, a.name == "*"))
    return items

Parsing captures `@patch` receiver annotations and import aliases while skipping cell magics.

In [ ]:
sample = mk_cell("#| export\nfrom pathlib import Path as P\nclass Child(Base): pass\n@delegates(run)\n@patch\ndef label(self: Box, **kwargs): return self.x")
defs, imps = cell_defs(sample), cell_imports(sample)
test_eq([(d.name, d.kind, d.receiver, d.bases, d.delegates_to) for d in defs],
    [("Child", "class", "", ("Base",), ""), ("label", "patch", "Box", (), "run")])
test_eq([(i.module, i.name, i.alias) for i in imps], [("pathlib", "Path", "P")])
test_eq(parse_cell_ast(mk_cell("%%bash\necho hi")), (None, None))
defs, imps

## Chapters and cell text format

Literate nbdev notebooks organize each topic into an H2 (`##`) chapter. `heading_info` reads a cell's leading Markdown heading, and `Chapter` holds each chapter's cell span and exported public symbols.

In [ ]:
#| export
_HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*#*\s*$")

@dataclass
class Chapter:
    "One H2-delimited section of a notebook."
    title: str
    level: int
    start: int  # Inclusive cell index
    end: int  # Exclusive cell index
    cells: list = field(default_factory=list)
    defs: list[str] = field(default_factory=list)

    @property
    def cell_ids(self) -> list[str]: return [str(c.get("id", "")) for c in self.cells]

def heading_info(cell) -> tuple[int, str] | None:
    "Return `(level, title)` if `cell` is a Markdown cell starting with an ATX heading."
    if cell.get("cell_type") != "markdown": return None
    for line in cell_source(cell).splitlines():
        s = line.strip()
        if not s: continue
        m = _HEADING_RE.match(s)
        return (len(m.group(1)), m.group(2).strip()) if m else None
    return None

`chapters` groups everything before the first H2 into `Notebook head` and slices the rest at each H2 boundary. A notebook with no H2 headings becomes one `Notebook` chapter.

In [ ]:
#| export
def chapters(nb) -> list[Chapter]:
    "Split `nb` into H2 chapters, keeping leading cells before the first H2 in `Notebook head`."
    cells = list(nb.cells)
    starts = [(i, h[1]) for i, c in enumerate(cells) if (h := heading_info(c)) and h[0] == 2]
    if not starts:
        defs = [d.name for c in cells if is_exported(c) for d in cell_defs(c) if d.is_public]
        return [Chapter("Notebook", 1, 0, len(cells), cells, defs)]
    spans = [("Notebook head", 1, 0, starts[0][0])] if starts[0][0] > 0 else []
    for pos, (idx, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        spans.append((title, 2, idx, end))
    out = []
    for title, lvl, s, e in spans:
        chunk = cells[s:e]
        defs = [d.name for c in chunk if is_exported(c) for d in cell_defs(c) if d.is_public]
        out.append(Chapter(title, lvl, s, e, chunk, defs))
    return out

`parse_cells` turns a compact text sketch (`%%markdown`, `%%code`, and `---` separators) into `NbCell` objects with stable IDs.

In [ ]:
#| export
def parse_cells(text: str) -> list:
    "Parse `%%markdown` / `%%code` blocks separated by `---` lines into `NbCell` objects."
    cells = []
    for block in re.split(r"(?m)^---\s*$", text.strip()):
        lines = block.strip().splitlines()
        if not lines: continue
        head = lines[0].strip().lower()
        if head in ("%%markdown", "%%md"): ctype, body = "markdown", "\n".join(lines[1:]).strip()
        elif head == "%%code": ctype, body = "code", "\n".join(lines[1:]).strip()
        elif head == "%%raw": ctype, body = "raw", "\n".join(lines[1:]).strip()
        else: ctype, body = "code", "\n".join(lines).strip()
        cells.append(mk_cell(body, cell_type=ctype))
    return cells

Parsing a short sketch and splitting it by chapters exposes each chapter's exported symbols.

In [ ]:
sketch = parse_cells("%%markdown\n# Billing\n---\n%%markdown\n## Totals\n---\n%%code\n#| export\ndef subtotal(items): return sum(items)\n---\n%%code\nassert subtotal([2, 3]) == 5")
chaps = chapters(new_nb(sketch))
test_eq([(c.title, c.defs, len(c.cells)) for c in chaps], [("Notebook head", [], 1), ("Totals", ["subtotal"], 3)])
[(c.title, c.defs) for c in chaps]

## Project notebooks and source ownership

`notebook_paths` discovers project notebooks while skipping build folders, hidden directories, and Git worktrees. `project_root` locates the enclosing `pyproject.toml` or `.git` directory.

In [ ]:
#| export
_SKIP_DIRS = frozenset({".git", ".venv", "venv", ".worktrees", "_proc", "_docs", "_site", ".ipynb_checkpoints", "__pycache__", "node_modules", "build", "dist"})
_AUTOGEN_RE = re.compile(r"^#\s*AUTOGENERATED!\s*DO NOT EDIT!", re.MULTILINE)
_CELL_SRC_RE = re.compile(r"^#\s*%%\s+(.+?\.ipynb)(?:\s|$)", re.MULTILINE)

def _skip_dir(name: str) -> bool: return name in _SKIP_DIRS or name.startswith(".") or name.endswith(".egg-info")

def _walk_ext(p: Path, suffix: str) -> list[Path]:
    if p.is_file(): return [p] if p.suffix == suffix else []
    if not p.exists(): return []
    found = []
    for cur, dirs, files in os.walk(p):
        dirs[:] = sorted(d for d in dirs if not _skip_dir(d))
        for fname in sorted(files):
            if fname.endswith(suffix): found.append(Path(cur) / fname)
    return found

def notebook_paths(path: str | Path = ".") -> list[Path]:
    "Return sorted `.ipynb` paths under `path`, skipping build, VCS, and worktree folders."
    return _walk_ext(Path(path).expanduser(), ".ipynb")

In [ ]:
#| export
def project_root(path: str | Path = ".") -> Path:
    "Find the nearest ancestor containing `pyproject.toml` or `.git`, else return `path`'s directory."
    p = Path(path).expanduser().resolve()
    cur = p.parent if p.is_file() else p
    for candidate in (cur, *cur.parents):
        if (candidate / "pyproject.toml").is_file() or (candidate / ".git").exists(): return candidate
    return cur

def package_name(path: str | Path = ".") -> str | None:
    "Return the configured nbdev package directory name for `path`, or `None`."
    root = project_root(path)
    cfg_path = root / "pyproject.toml"
    if not cfg_path.is_file(): return None
    cfg = tomllib.loads(cfg_path.read_text(encoding="utf-8"))
    nb_cfg = cfg.get("tool", {}).get("nbdev", {})
    pkg = nb_cfg.get("lib_path") or cfg.get("project", {}).get("name", "").replace("-", "_")
    return pkg or None

`generated_owner` inspects a `.py` file header and returns its owning `.ipynb` only when nbdev generated it, returning `None` for hand-written Python. `notebook_export_targets` resolves every module target (`#| default_exp` and `#| export <mod>`), while `handwritten_py_paths` and `generated_py_paths` separate manual code from generated modules.

In [ ]:
#| export
def is_generated_py(py_path: str | Path) -> bool:
    "True when `py_path` carries nbdev's `# AUTOGENERATED!` banner."
    p = Path(py_path).expanduser()
    if p.suffix != ".py" or not p.is_file(): return False
    return bool(_AUTOGEN_RE.search(p.read_text(encoding="utf-8", errors="replace")[:32768]))

def generated_owner_ref(py_path: str | Path) -> Path | None:
    "Return the `.ipynb` path referenced by an nbdev-generated `.py` header (even if missing)."
    p = Path(py_path).expanduser()
    if p.suffix != ".py" or not p.is_file(): return None
    head = p.read_text(encoding="utf-8", errors="replace")[:32768]
    if not _AUTOGEN_RE.search(head): return None
    m = _CELL_SRC_RE.search(head)
    return (p.parent / m.group(1)).resolve() if m else None

def generated_owner(py_path: str | Path) -> Path | None:
    "Return the owning `.ipynb` for an nbdev-generated `.py` file, or `None` if hand-written."
    owner = generated_owner_ref(py_path)
    return owner if owner is not None and owner.is_file() else None

`exported_py_path` and `notebook_export_targets` resolve `#| default_exp` and `#| export <module>` targets, while `generated_py_paths` and `handwritten_py_paths` separate generated modules from manual Python files.

In [ ]:
#| export
def _module_py_path(root: Path, pkg: str, mod: str) -> Path: return root / pkg / Path(*mod.split(".")).with_suffix(".py")

def exported_py_path(nb_path: str | Path, nb=None) -> Path | None:
    "Return the generated `.py` path declared by `#| default_exp` in `nb_path`, or `None`."
    p = Path(nb_path).expanduser().resolve()
    nb = nb or read_nb(p)
    mod = next((cell_directives(c)["default_exp"] for c in nb.cells if "default_exp" in cell_directives(c)), None)
    if not mod: return None
    root = project_root(p)
    pkg = package_name(root)
    return _module_py_path(root, pkg, mod) if pkg else None

def notebook_export_targets(nb_path: str | Path, nb=None) -> dict[str, Path]:
    "Return `{module: py_path}` for every target module `nb_path` exports to (`default_exp` and `#| export <mod>`)."
    p = Path(nb_path).expanduser().resolve()
    nb = nb or read_nb(p)
    root = project_root(p)
    pkg = package_name(root)
    if not pkg: return {}
    default_exp = next((cell_directives(c)["default_exp"] for c in nb.cells if "default_exp" in cell_directives(c)), None)
    targets = {default_exp: _module_py_path(root, pkg, default_exp)} if default_exp else {}
    for cell in nb.cells:
        mod = cell_export_module(cell, default_exp)
        if mod and mod not in targets: targets[mod] = _module_py_path(root, pkg, mod)
    return targets

In [ ]:
#| export
def _project_py_paths(path: str | Path = ".") -> list[Path]: return _walk_ext(Path(path).expanduser(), ".py")

def generated_py_paths(path: str | Path = ".") -> list[Path]:
    "Return sorted nbdev-generated `.py` files under `path` (excluding `_modidx.py`)."
    return [f for f in _project_py_paths(path) if f.name != "_modidx.py" and is_generated_py(f)]

def handwritten_py_paths(path: str | Path = ".") -> list[Path]:
    "Return sorted hand-written `.py` files under `path` (excluding generated modules and `_modidx.py`)."
    return [f for f in _project_py_paths(path) if f.name != "_modidx.py" and not is_generated_py(f)]

A disposable project with one generated module and one hand-written file verifies both ownership directions.

In [ ]:
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "pyproject.toml").write_text('[project]\nname = "demo-pkg"\n')
    for d in ("nbs", "demo_pkg", "tests"): (root / d).mkdir()
    nb_file, gen_py, extra_py, hand_py = root / "nbs/00_math.ipynb", root / "demo_pkg/calc.py", root / "demo_pkg/extra.py", root / "tests/test_calc.py"
    write_nb(new_nb([mk_cell("#| default_exp calc"), mk_cell("#| export extra\nx = 1")]), nb_file)
    gen_py.write_text("# AUTOGENERATED! DO NOT EDIT!\n# %% ../nbs/00_math.ipynb #abcd1234\nx = 1\n")
    hand_py.write_text("assert True\n")
    test_eq((generated_owner(gen_py), generated_owner(hand_py), exported_py_path(nb_file), notebook_export_targets(nb_file)),
        (nb_file.resolve(), None, gen_py.resolve(), {"calc": gen_py.resolve(), "extra": extra_py.resolve()}))
    test_eq((generated_py_paths(root), handwritten_py_paths(root)), ([gen_py], [hand_py]))